# 159.224 Week 10 Tutorial: Self Attention in Matrix and PyTorch Form


## Main question for this week

> How do we calculate self attention for **all tokens at once**?

Week 9 tutorial followed the lecture's one token view.

Week 10 tutorial follows the lecture's matrix form.

This notebook uses exactly the same convention as the slides:

* individual token vectors $x_i,q_i,k_i,v_i,z_i$ are **column vectors**;
* when we build matrices, each token vector is **transposed** and stored as one row.

## Very Short Week 9 Recap

For one token $i$:

$$
s_{ij}
=
\frac{q_i^\top k_j}{\sqrt{d_k}},
$$

then:

$$
a_{ij}
=
\frac{\exp(s_{ij})}
{\sum_{m=1}^{n}\exp(s_{im})},
$$

and:

$$
z_i
=
\sum_{j=1}^{n}a_{ij}v_j.
$$

Week 10 performs these same calculations for all tokens at once.


In [ ]:
import math
import torch
import torch.nn as nn

torch.manual_seed(0)

print("PyTorch version:", torch.__version__)


PyTorch version: 2.10.0+cpu


## Part 1: From one token to the whole sequence

### Important: the representation changes in Week 10

In Week 9, we focused on **one query only**. To make that calculation easy to see, we temporarily placed the key and value vectors side by side as columns.

In Week 10, our goal is different: we want to calculate attention for **all tokens at once**.

Therefore, we now follow the formal matrix representation used in the lecture.

From this point onward:

> **one row = one token**

This is the formal matrix convention used for $X$, $Q$, $K$, $V$, and $Z$.

### Step 1: Stack token vectors as rows

In the lecture, every $x_i$ is a column vector:

$$
x_i
\in
\mathbb{R}^{d_{\mathrm{model}}\times 1}.
$$

Therefore:

$$
x_i^\top
\in
\mathbb{R}^{1\times d_{\mathrm{model}}}.
$$

To represent the whole sequence, we stack the transposed token vectors as rows:

$$
X
=
\begin{bmatrix}
x_1^\top\\
x_2^\top\\
\vdots\\
x_n^\top
\end{bmatrix}
=
\left[
\begin{array}{c}
\rule{0.7cm}{0.4pt}\;\;x_1^\top\;\;\rule{0.7cm}{0.4pt}\\
\rule{0.7cm}{0.4pt}\;\;x_2^\top\;\;\rule{0.7cm}{0.4pt}\\
\vdots\\
\rule{0.7cm}{0.4pt}\;\;x_n^\top\;\;\rule{0.7cm}{0.4pt}
\end{array}
\right]
\in
\mathbb{R}^{n\times d_{\mathrm{model}}}.
$$

The visual meaning is:

* row 1 is $x_1^\top$
* row 2 is $x_2^\top$
* $\ldots$
* row $n$ is $x_n^\top$

So:

> **one row of $X$ = one token**

### Step 2: Produce all Queries, Keys, and Values

The lecture matrix form is:

$$
Q=XW_Q,
\qquad
K=XW_K,
\qquad
V=XW_V.
$$

with:

$$
W_Q,W_K
\in
\mathbb{R}^{d_{\mathrm{model}}\times d_k},
$$

and:

$$
W_V
\in
\mathbb{R}^{d_{\mathrm{model}}\times d_v}.
$$

Therefore:

$$
Q,K
\in
\mathbb{R}^{n\times d_k},
\qquad
V
\in
\mathbb{R}^{n\times d_v}.
$$

Now look carefully at how these matrices are organised.

Each row of $Q$ is one query vector written in transposed form:

$$
Q
=
\begin{bmatrix}
q_1^\top\\
q_2^\top\\
\vdots\\
q_n^\top
\end{bmatrix}
=
\left[
\begin{array}{c}
\rule{0.7cm}{0.4pt}\;\;q_1^\top\;\;\rule{0.7cm}{0.4pt}\\
\rule{0.7cm}{0.4pt}\;\;q_2^\top\;\;\rule{0.7cm}{0.4pt}\\
\vdots\\
\rule{0.7cm}{0.4pt}\;\;q_n^\top\;\;\rule{0.7cm}{0.4pt}
\end{array}
\right]
\in
\mathbb{R}^{n\times d_k}.
$$

Each row of $K$ is one key vector written in transposed form:

$$
K
=
\begin{bmatrix}
k_1^\top\\
k_2^\top\\
\vdots\\
k_n^\top
\end{bmatrix}
=
\left[
\begin{array}{c}
\rule{0.7cm}{0.4pt}\;\;k_1^\top\;\;\rule{0.7cm}{0.4pt}\\
\rule{0.7cm}{0.4pt}\;\;k_2^\top\;\;\rule{0.7cm}{0.4pt}\\
\vdots\\
\rule{0.7cm}{0.4pt}\;\;k_n^\top\;\;\rule{0.7cm}{0.4pt}
\end{array}
\right]
\in
\mathbb{R}^{n\times d_k}.
$$

Each row of $V$ is one value vector written in transposed form:

$$
V
=
\begin{bmatrix}
v_1^\top\\
v_2^\top\\
\vdots\\
v_n^\top
\end{bmatrix}
=
\left[
\begin{array}{c}
\rule{0.7cm}{0.4pt}\;\;v_1^\top\;\;\rule{0.7cm}{0.4pt}\\
\rule{0.7cm}{0.4pt}\;\;v_2^\top\;\;\rule{0.7cm}{0.4pt}\\
\vdots\\
\rule{0.7cm}{0.4pt}\;\;v_n^\top\;\;\rule{0.7cm}{0.4pt}
\end{array}
\right]
\in
\mathbb{R}^{n\times d_v}.
$$

So the same row based convention is used throughout:

```text
X : one row = one input token
Q : one row = one Query
K : one row = one Key
V : one row = one Value
```

This is exactly the same projection idea as Week 9.

The difference is that in Week 10 we organise **all tokens together in matrices** so that attention for the entire sequence can be calculated at once.







### Step 3: Calculate all Query and Key scores

The lecture defines:

$$
S
=
\frac{QK^\top}{\sqrt{d_k}}.
$$

Before multiplying, look at the shapes:

$$
Q
\in
\mathbb{R}^{n\times d_k},
$$

while:

$$
K^\top
\in
\mathbb{R}^{d_k\times n}.
$$

Since:

$$
K
=
\begin{bmatrix}
k_1^\top\\
k_2^\top\\
\vdots\\
k_n^\top
\end{bmatrix},
$$

transposing it gives:

$$
K^\top
=
\begin{bmatrix}
| & | & & |\\
k_1 & k_2 & \cdots & k_n\\
| & | & & |
\end{bmatrix}.
$$

Therefore:

$$
QK^\top
=
\left[
\begin{array}{c}
\rule{0.7cm}{0.4pt}\;\;q_1^\top\;\;\rule{0.7cm}{0.4pt}\\
\rule{0.7cm}{0.4pt}\;\;q_2^\top\;\;\rule{0.7cm}{0.4pt}\\
\vdots\\
\rule{0.7cm}{0.4pt}\;\;q_n^\top\;\;\rule{0.7cm}{0.4pt}
\end{array}
\right]
\begin{bmatrix}
| & | & & |\\
k_1 & k_2 & \cdots & k_n\\
| & | & & |
\end{bmatrix}.
$$

Now the structure becomes very clear:

$$
QK^\top
=
\begin{bmatrix}
q_1^\top k_1 & q_1^\top k_2 & \cdots & q_1^\top k_n\\
q_2^\top k_1 & q_2^\top k_2 & \cdots & q_2^\top k_n\\
\vdots & \vdots & \ddots & \vdots\\
q_n^\top k_1 & q_n^\top k_2 & \cdots & q_n^\top k_n
\end{bmatrix}.
$$

After scaling:

$$
S
=
\frac{1}{\sqrt{d_k}}
\begin{bmatrix}
q_1^\top k_1 & q_1^\top k_2 & \cdots & q_1^\top k_n\\
q_2^\top k_1 & q_2^\top k_2 & \cdots & q_2^\top k_n\\
\vdots & \vdots & \ddots & \vdots\\
q_n^\top k_1 & q_n^\top k_2 & \cdots & q_n^\top k_n
\end{bmatrix}.
$$

The shape is:

$$
(n\times d_k)(d_k\times n)
\rightarrow
(n\times n).
$$

Each entry is:

$$
S_{ij}
=
\frac{q_i^\top k_j}{\sqrt{d_k}}.
$$

So:

* row $i$ contains all scores produced by query $q_i$
* column $j$ corresponds to key $k_j$
* entry $S_{ij}$ is the relevance score between query $q_i$ and key $k_j$

This is the key connection between Week 9 and Week 10.

In Week 9, we calculated:

$$
q_i^\top k_1,
\quad
q_i^\top k_2,
\quad
\ldots,
\quad
q_i^\top k_n
$$

for **one query**.

In Week 10, $QK^\top$ calculates the same thing for **every query at once**.

In [ ]:
# Each row of Q stores q_i^T.
Q = torch.tensor([
    [1.0, 0.0],
    [0.0, 1.0],
    [1.0, 1.0]
])

# Each row of K stores k_i^T.
K = torch.tensor([
    [1.0, 0.0],
    [0.0, 1.0],
    [1.0, 1.0]
])

# Each row of V stores v_i^T.
V = torch.tensor([
    [1.0, 0.0, 0.0],
    [0.0, 1.0, 0.0],
    [0.0, 0.0, 1.0]
])

# S = Q K^T / sqrt(d_k)
S = Q @ K.transpose(-2, -1)    # this line is very very important ;) try your best to understand .transpose() method
S = S / math.sqrt(K.shape[-1])

# A = softmax_row(S)
A = torch.softmax(S, dim=-1)  # this line is very very important ;) try your best to understand dim=-1

# Z = A V
Z = A @ V

print("Q shape:", Q.shape)
print("K shape:", K.shape)
print("V shape:", V.shape)
print("S shape:", S.shape)
print("A shape:", A.shape)
print("Z shape:", Z.shape)


Q shape: torch.Size([3, 2])
K shape: torch.Size([3, 2])
V shape: torch.Size([3, 3])
S shape: torch.Size([3, 3])
A shape: torch.Size([3, 3])
Z shape: torch.Size([3, 3])


### Important: these Q, K, and V values above are only example values

These numbers are **manually chosen only to make the calculation easy to follow**.

They are **not** special values used by real self attention models.

In a real model, $Q$, $K$, and $V$ are produced from the input data through learned projection matrices:

$$
Q=XW_Q,
\qquad
K=XW_K,
\qquad
V=XW_V.
$$

The projection matrices:

$$
W_Q,
\qquad
W_K,
\qquad
W_V
$$

are learned during training.

Therefore, the actual entries of $Q$, $K$, and $V$ are learned from the data and can take many different real values, for example:

$$
0.37,\quad -1.24,\quad 2.08,\quad 0.006,\quad -0.81,
$$

or many other values.

They are not restricted to simple numbers such as:

$$
0,\quad 0.1,\quad 1.
$$

We use simple numbers in this tutorial only so that you can clearly see the matrix calculation:

$$
QK^\top
\rightarrow
\text{scaling}
\rightarrow
\text{softmax}
\rightarrow
AV.
$$

> **The purpose of this example is to understand the attention calculation, not to reproduce the actual learned values of a trained model.**

### The complete shape flow

This is the same matrix flow shown in the lecture.

<table style="width:100%;">
  <tr>
    <th style="text-align:left;">Tensor</th>
    <th style="text-align:left;">Meaning</th>
    <th style="text-align:left; white-space:nowrap;">Shape</th>
  </tr>
  <tr>
    <td style="white-space:nowrap;">\(X\)</td>
    <td>input sequence</td>
    <td style="white-space:nowrap;">\((n,d_{\mathrm{model}})\)</td>
  </tr>
  <tr>
    <td style="white-space:nowrap;">\(Q\)</td>
    <td>all queries</td>
    <td style="white-space:nowrap;">\((n,d_k)\)</td>
  </tr>
  <tr>
    <td style="white-space:nowrap;">\(K\)</td>
    <td>all keys</td>
    <td style="white-space:nowrap;">\((n,d_k)\)</td>
  </tr>
  <tr>
    <td style="white-space:nowrap;">\(V\)</td>
    <td>all values</td>
    <td style="white-space:nowrap;">\((n,d_v)\)</td>
  </tr>
  <tr>
    <td style="white-space:nowrap;">\(S=QK^\top/\sqrt{d_k}\)</td>
    <td>attention scores</td>
    <td style="white-space:nowrap;">\((n,n)\)</td>
  </tr>
  <tr>
    <td style="white-space:nowrap;">\(A=\mathrm{softmax}_{\mathrm{row}}(S)\)</td>
    <td>attention weights</td>
    <td style="white-space:nowrap;">\((n,n)\)</td>
  </tr>
  <tr>
    <td style="white-space:nowrap;">\(Z=AV\)</td>
    <td>attention output</td>
    <td style="white-space:nowrap;">\((n,d_v)\)</td>
  </tr>
</table>

The two important matrix multiplications are:

$$
(n\times d_k)(d_k\times n)
\rightarrow
(n\times n),
$$

and:

$$
(n\times n)(n\times d_v)
\rightarrow
(n\times d_v).
$$

### Shape checkpoint

Suppose:

$$
Q,K\in\mathbb{R}^{6\times4},
\qquad
V\in\mathbb{R}^{6\times5}.
$$

What are the shapes of:

1. $S=QK^\top/\sqrt{d_k}$?
2. $A=\mathrm{softmax}_{\mathrm{row}}(S)$?
3. $Z=AV$?

<details>
<summary><b>Check answer</b></summary>

Because:

$$
(6\times4)(4\times6)
\rightarrow
(6\times6),
$$

we have:

$$
S\in\mathbb{R}^{6\times6}.
$$

Row wise softmax does not change the shape:

$$
A\in\mathbb{R}^{6\times6}.
$$

Finally:

$$
(6\times6)(6\times5)
\rightarrow
(6\times5),
$$

so:

$$
Z\in\mathbb{R}^{6\times5}.
$$

</details>


## Part 2: Add the batch dimension $B$

So far, we have described one sequence at a time.

For one sequence, the input has shape:

$$
(T,d_{\mathrm{model}}),
$$

where:

* $T$ is the sequence length;
* $d_{\mathrm{model}}$ is the feature dimension of each token.

In PyTorch, however, we usually process several sequences together in one **batch**.

So we add one new dimension at the front:

$$
(B,T,d_{\mathrm{model}}),
$$

where $B$ is the batch size.

The lecture uses $n$ for sequence length.

In the PyTorch code below, we use $T$ for the same quantity:

$$
T=n.
$$

### What does the batch dimension mean?

Suppose:

$$
X:(B,T,d_{\mathrm{model}}).
$$

You can think of $X$ as containing $B$ separate input samples:

```text
X[0, :, :]  -> first sample
X[1, :, :]  -> second sample
X[2, :, :]  -> third sample
...
```

For example:

```python
first_sample = X[0, :, :]
```

has shape:

$$
(T,d_{\mathrm{model}}).
$$

Similarly:

```python
second_sample = X[1, :, :]
```

also has shape:

$$
(T,d_{\mathrm{model}}).
$$

So the batch dimension simply means:

> **we are processing several sequences at the same time.**

The self attention calculation is applied independently to each sample in the batch.

### The same idea applies to $Q$, $K$, and $V$

After the projection layers:

$$
Q,K:(B,T,d_k),
\qquad
V:(B,T,d_v).
$$

For example:

```python
Q[0, :, :]
```

is the Query matrix for the first sample:

$$
Q^{(0)}:(T,d_k),
$$

and:

```python
K[0, :, :]
```

is the Key matrix for the first sample:

$$
K^{(0)}:(T,d_k).
$$

Similarly:

```python
Q[1, :, :]
K[1, :, :]
```

belong to the second sample.

So conceptually:

```text
sample 0: Q[0] works with K[0]
sample 1: Q[1] works with K[1]
sample 2: Q[2] works with K[2]
...
```

The samples do **not** mix with each other during this matrix multiplication.


### Score tensor

For each sample, we still want:

$$
QK^\top.
$$

But now the calculation is performed for all $B$ samples together.

The score tensor is:

$$
S
=
\frac{QK^\top}{\sqrt{d_k}}
:
(B,T,T).
$$

In PyTorch, we transpose the last two dimensions of $K$:

```python
K.transpose(-2, -1)
```

so:

```text
K                  : (B, T, d_k)
K.transpose(-2,-1) : (B, d_k, T)
```

Then:

```text
Q @ K.transpose(-2,-1)

(B, T, d_k) @ (B, d_k, T)

-> (B, T, T)
```

You can think of this as PyTorch performing:

```text
Q[0] @ K[0].T
Q[1] @ K[1].T
Q[2] @ K[2].T
...
```

for all samples in the batch.

So each sample gets its own attention score matrix:

$$
(T,T).
$$

The batch simply collects all of them together:

$$
(B,T,T).
$$

### Row wise softmax

The last dimension of $S$ contains the key positions.

Therefore:

```python
torch.softmax(S, dim=-1)
```

applies softmax over the keys for each query, independently for every sample in the batch.

This is the batched PyTorch version of the same row wise softmax used in the lecture.

## Part 3: The complete PyTorch calculation



Now look at the whole calculation in one place.

The code follows the lecture formulas:

$$
Q=XW_Q,
\qquad
K=XW_K,
\qquad
V=XW_V,
$$

$$
S=\frac{QK^\top}{\sqrt{d_k}},
$$

$$
A=\mathrm{softmax}_{\mathrm{row}}(S),
$$

$$
Z=AV.
$$

In [ ]:
class SimpleSelfAttention(nn.Module):
    def __init__(self, d_model=8, d_k=3, d_v=5):
        super().__init__()

        # Lecture: W_Q and W_K map d_model to d_k.
        # bias=False keeps the code exactly aligned with Q = XW_Q and K = XW_K.
        self.q_proj = nn.Linear(d_model, d_k, bias=False)
        self.k_proj = nn.Linear(d_model, d_k, bias=False)

        # Lecture: W_V maps d_model to d_v.
        self.v_proj = nn.Linear(d_model, d_v, bias=False)

        self.d_k = d_k

    def forward(self, x):
        # PyTorch shape: (B, T, d_model)
        # T plays the same role as n in the lecture.

        # Q = XW_Q, K = XW_K, V = XW_V
        Q = self.q_proj(x)   # (B, T, d_k)
        K = self.k_proj(x)   # (B, T, d_k)
        V = self.v_proj(x)   # (B, T, d_v)

        # S = QK^T / sqrt(d_k)
        S = Q @ K.transpose(-2, -1)   # (B, T, T)
        S = S / math.sqrt(self.d_k)

        # A = softmax_row(S)
        A = torch.softmax(S, dim=-1)   # (B, T, T)

        # Z = AV
        Z = A @ V   # (B, T, d_v)

        return Z


In [ ]:
model = SimpleSelfAttention(d_model=8, d_k=3, d_v=5)

# B=2, T=4, d_model=8
x = torch.randn(2, 4, 8)

Z = model(x)

print("input shape:", x.shape)
print("Z shape:    ", Z.shape)


input shape: torch.Size([2, 4, 8])
Z shape:     torch.Size([2, 4, 5])


### Read the implementation as five checks

When you read self attention code, connect every line back to the lecture formulas.

### 1. Projection dimensions

Lecture:

$$
Q=XW_Q,
\qquad
K=XW_K,
\qquad
V=XW_V.
$$

Therefore Q and K end in $d_k$, while V ends in $d_v$.

### 2. Score matrix

Lecture:

$$
S
=
\frac{QK^\top}{\sqrt{d_k}}.
$$

PyTorch:

```python
S = Q @ K.transpose(-2, -1)
S = S / math.sqrt(d_k)
```

### 3. Row wise softmax

Lecture:

$$
A
=
\mathrm{softmax}_{\mathrm{row}}(S).
$$

PyTorch:

```python
A = torch.softmax(S, dim=-1)
```

### 4. Weighted sum of values

Lecture:

$$
Z=AV.
$$

PyTorch:

```python
Z = A @ V
```

### 5. Expected shapes

```text
Q : (B, T, d_k)
K : (B, T, d_k)
V : (B, T, d_v)
S : (B, T, T)
A : (B, T, T)
Z : (B, T, d_v)
```


## Part 4: Practice Code Correction

The following program is intended to implement single head self attention.

The intended lecture formulas are:

$$
Q=XW_Q,
\qquad
K=XW_K,
\qquad
V=XW_V,
$$

$$
S
=
\frac{QK^\top}{\sqrt{d_k}},
\qquad
A
=
\mathrm{softmax}_{\mathrm{row}}(S),
\qquad
Z=AV.
$$

The input has shape:

$$
(B,T,d_{\mathrm{model}})
$$

and the expected output has shape:

$$
(B,T,d_v).
$$

The code contains **exactly five errors**.

The style and difficulty are similar to the code reading expected in the exam.

Identify each incorrect line and give a correct replacement.

```python
import math
import torch
import torch.nn as nn

class PracticeSelfAttention(nn.Module):
    def __init__(self, d_model=24, d_k=8, d_v=12):
        super().__init__()

        self.d_k = d_k
        self.d_v = d_v

        self.q_proj = nn.Linear(d_model, d_k, bias=False)
        self.k_proj = nn.Linear(d_model, d_k, bias=False)
        self.v_proj = nn.Linear(d_model, d_k, bias=False)

    def forward(self, x):
        Q = self.q_proj(x)
        K = self.k_proj(x)
        V = self.v_proj(x)

        S = Q.transpose(-2, -1) @ K
        S = S / math.sqrt(self.d_v)

        A = torch.softmax(S, dim=-1)

        Z = A @ K

        return A
```


### Answer and Explanation

<details>
<summary><b>Show the five corrections</b></summary>

### Correction 1: Value projection dimension

Incorrect:

```python
self.v_proj = nn.Linear(d_model, d_k, bias=False)
```

Correct:

```python
self.v_proj = nn.Linear(d_model, d_v, bias=False)
```

The lecture defines:

$$
V=XW_V
\in
\mathbb{R}^{n\times d_v}.
$$

### Correction 2: Score matrix

Incorrect:

```python
S = Q.transpose(-2, -1) @ K
```

Correct:

```python
S = Q @ K.transpose(-2, -1)
```

This implements:

$$
QK^\top.
$$

### Correction 3: Scaling dimension

Incorrect:

```python
S = S / math.sqrt(self.d_v)
```

Correct:

```python
S = S / math.sqrt(self.d_k)
```

The lecture uses:

$$
\frac{QK^\top}{\sqrt{d_k}}.
$$

### Correction 4: Weighted sum

Incorrect:

```python
Z = A @ K
```

Correct:

```python
Z = A @ V
```

The lecture output is:

$$
Z=AV.
$$

### Correction 5: Returned tensor

Incorrect:

```python
return A
```

Correct:

```python
return Z
```

$A$ contains the attention weights.

$Z$ contains the new contextual representations.

</details>
